## Data Preparation

In [22]:
import pandas as pd

# change this path to the folder where you saved the file
data = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(data)

### Reading the data

In [23]:
df.shape

(7043, 21)

In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [25]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [26]:
df.head().T

,0,1,2,3,4
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
SeniorCitizen,0,0,0,0,0
Partner,Yes,No,No,No,No
Dependents,No,No,No,No,No
tenure,1,34,2,45,2
PhoneService,No,Yes,Yes,No,Yes
MultipleLines,No phone service,No,No,No phone service,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic
OnlineSecurity,No,Yes,Yes,Yes,No


### Making column names and values look uniform

In [27]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.select_dtypes(exclude='number').columns)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

df.head(3).T

,0,1,2
customerid,7590-vhveg,5575-gnvde,3668-qpybk
gender,female,male,male
seniorcitizen,0,0,0
partner,yes,no,no
dependents,no,no,no
tenure,1,34,2
phoneservice,no,yes,yes
multiplelines,no_phone_service,no,no
internetservice,dsl,dsl,dsl
onlinesecurity,no,yes,yes


### Fixing totalcharges

Two things are interesting here:

- `seniorcitizen` is stored as a number (`0` or `1`) instead of text (`yes` or `no`). That is fine for us.
- `totalcharges` is stored as text, but a total charge should be a number. Something is wrong.

In [28]:
pd.to_numeric(df.totalcharges)
# Output: ValueError: Unable to parse string "_" at position 488

ValueError: Unable to parse string "_" at position 488

In [29]:
# tc means total charges
tc = pd.to_numeric(df.totalcharges, errors='coerce')

In [30]:
print(df['totalcharges'].isnull().sum())
# Output: np.int64(11)

df[tc.isnull()][['customerid', 'tenure', 'totalcharges']]

0


,customerid,tenure,totalcharges
488,4472-lvygi,0,_
753,3115-czmzd,0,_
936,5709-lvoeq,0,_
1082,4367-nuyao,0,_
1340,1371-dwpaz,0,_
3331,7644-omvmy,0,_
3826,3213-vvolg,0,_
4380,2520-sgtta,0,_
5218,2923-arzlg,0,_
6670,4075-wkniu,0,_


In [31]:
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

df.totalcharges.isnull().sum()
# Output: np.int64(0)

np.int64(0)

It is 0 for all 11 customers. They are brand-new customers who haven't paid anything yet, so their total charge really is 0. That makes filling the missing values with 0 a good choice here. Keep in mind that 0 is not always the right choice; here it matches what the data tells us.

In [32]:
df.totalcharges.dtype

dtype('float64')

### Checking if the churn variable needs any preparation

In [33]:
df.churn.head()

0     no
1     no
2    yes
3     no
4    yes
Name: churn, dtype: str

In [34]:
(df.churn == 'yes').head()

0    False
1    False
2     True
3    False
4     True
Name: churn, dtype: bool

In [35]:
df.churn = (df.churn == 'yes').astype(int)
df.head()

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,7590-vhveg,female,0,yes,no,1,no,no_phone_service,dsl,no,...,no,no,no,no,month-to-month,yes,electronic_check,29.85,29.85,0
1,5575-gnvde,male,0,no,no,34,yes,no,dsl,yes,...,yes,no,no,no,one_year,no,mailed_check,56.95,1889.50,0
2,3668-qpybk,male,0,no,no,2,yes,no,dsl,yes,...,no,no,no,no,month-to-month,yes,mailed_check,53.85,108.15,1
3,7795-cfocw,male,0,no,no,45,no,no_phone_service,dsl,yes,...,yes,yes,no,no,one_year,no,bank_transfer_(automatic),42.30,1840.75,0
4,9237-hqitu,female,0,no,no,2,yes,no,fiber_optic,no,...,no,no,no,no,month-to-month,yes,electronic_check,70.70,151.65,1


In [16]:
df.isnull().sum().sum()

np.int64(0)

------

## Setting up the Validation Framework
Before we explore the data, we put part of it aside. Imagine studying for an exam using the exact exam questions. You would get a great grade, but you wouldn't know whether you really understood the material. A model has the same problem, so we split the data into two parts:

- **Train** (80%): we explore it, and the model learns from it.
- **Validation** (20%): we hide it for now and use it later to check how good the model is on customers it has never seen.

We use scikit-learn's `train_test_split` to do that.

In [17]:
from sklearn.model_selection import train_test_split

df_train, df_val = train_test_split(df, test_size=0.2, random_state=42)

len(df_train), len(df_val)
# Output: (5634, 1409)

/home/omaramir/projects/ml_autumn_2027/.venv/lib/python3.13/site-packages/joblib/_multiprocessing_helpers.py:44: UserWarning: [Errno 13] Permission denied.  joblib will operate in serial mode
  warnings.warn("%s.  joblib will operate in serial mode" % (e,))


(5634, 1409)

- `test_size=0.2` sends 20% of the customers to validation.
- `train_test_split` shuffles the rows randomly before splitting. `random_state=42` fixes that randomness, so everyone gets the same split every time. We will use the same line in the next section.

**From now on, we only explore `df_train`.**

## EDA – Exploratory Data Analysis

### 1. Looking at the target variable (churn)

First, how many customers are churning and how many are not?

In [18]:
df_train.churn.value_counts()


churn
0    4138
1    1496
Name: count, dtype: int64

Out of 5,634 customers in the training data, 1,496 left (churning) and 4,138 stayed (not churning). Knowing this balance is an important first step, because it affects how we judge the model later.

With `normalize=True` we get the share instead of the count. The share of churning customers is called the **churn rate**.

In [19]:
df_train.churn.value_counts(normalize=True)

churn
0    0.734469
1    0.265531
Name: proportion, dtype: float64

The churn rate is almost 27%. There is another way to get it: the mean() function.

In [20]:
global_churn = df_train.churn.mean()
global_churn

np.float64(0.2655307064252751)

### 2. Churn rate inside a group
The global churn rate (almost 27%) is one number for all customers. But are all customers the same? Maybe some kinds of customers leave much more than others. To find out, we look at the churn rate **inside a group** of customers and compare it with the global churn rate.

Let's take the `contract` column. First, which groups does it have?

In [21]:
df_train.contract.value_counts()

contract
month-to-month    3083
two_year          1339
one_year          1212
Name: count, dtype: int64

Three groups: customers who pay month by month, and customers who signed a one-year or a two-year contract.

#### Step 1: select one group

To select only the month-to-month customers, we first ask a yes or no question for every customer: "is your contract month-to-month?"

In [22]:
(df_train.contract == 'month-to-month').head()

2142    False
1623    False
6074     True
1362     True
6754    False
Name: contract, dtype: bool

Then we put this question inside df_train[...]. Pandas keeps only the rows where the answer is True.

In [23]:
month_to_month = df_train[df_train.contract == 'month-to-month']

len(month_to_month)
# Output: 3083

month_to_month[['customerid', 'contract', 'churn']].head()

,customerid,contract,churn
6074,3797-vtidr,month-to-month,1
1362,2568-brgyx,month-to-month,1
1212,4291-shsbh,month-to-month,0
6791,5204-qzxpu,month-to-month,1
5852,3398-zouaa,month-to-month,1


Now we have a smaller table with only the 3,083 month-to-month customers.

### Step 2: calculate the churn rate of that group

We already know the trick: the mean of the churn column is the churn rate.

In [24]:
churn_month_to_month = df_train[df_train.contract == 'month-to-month'].churn.mean()
churn_month_to_month

np.float64(0.4265325981187155)

About 43% of month-to-month customers left. Let's do the same for the other two groups:

In [25]:
churn_one_year = df_train[df_train.contract == 'one_year'].churn.mean()
churn_one_year
# Output: np.float64(0.11798679867986799)

churn_two_year = df_train[df_train.contract == 'two_year'].churn.mean()
churn_two_year
# Output: np.float64(0.028379387602688575)

np.float64(0.028379387602688575)

Step 3: compare with the global churn rate

month-to-month	43%

one_year	12%

two_year	3%


The rule for reading it:

- A group **above** the global churn rate leaves **more** than the average customer.
- A group **below** the global churn rate leaves **less** than the average customer.
- A group **close** to the global churn rate behaves like everyone else, so the column doesn't tell us much.

The groups here are very different from each other, so the contract tells us a lot about churn.

The global churn rate is one number for everyone. Now we split the customers into groups and look at the churn rate inside each group. Let's start with the contract type.

### A shortcut: groupby

We wrote one line of code for each group. With many groups that gets long. `groupby` does all of it in one line:

In [26]:
df_train.groupby('contract').churn.mean()

contract
month-to-month    0.426533
one_year          0.117987
two_year          0.028379
Name: churn, dtype: float64

The same three numbers we calculated by hand. `groupby('contract')` splits the customers into groups by their contract, and `.churn.mean()` calculates the churn rate inside each group.

We can also ask for the size of each group, with `agg`, which takes a list of calculations:

In [27]:
df_train.groupby('contract').churn.agg(['mean', 'count'])

,mean,count
contract,,
month-to-month,0.426533,3083
one_year,0.117987,1212
two_year,0.028379,1339


`count` matters because a churn rate from a big group is more reliable than one from a group of only a few customers.

Now let's use the shortcut on another column: the type of internet service.

In [28]:
df_train.groupby('internetservice').churn.agg(['mean', 'count'])

,mean,count
internetservice,,
dsl,0.191851,1939
fiber_optic,0.415558,2481
no,0.076606,1214


Fiber optic customers leave much more often than average (about 42%), while customers without internet service rarely leave (about 8%).

**Question for the class:** if you were the company, which customers would you send the discount to first?

We looked at two columns, but there are 16 categorical columns in total. Checking every table by eye is slow, and it doesn't tell us which column is more important overall. For that we need one number per column.

-------

## Feature importance
Feature importance tells us which columns (features) affect our target the most. We use two tools: mutual information for columns that hold categories, and correlation for columns that hold numbers.

### Defining numerical and categorical columns

In [29]:
numerical = ['tenure', 'monthlycharges', 'totalcharges']

categorical = ['gender', 'seniorcitizen', 'partner', 'dependents',
       'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod']

We leave out customerid, because it is just a label for each customer, and churn, because it is the answer we want to predict. seniorcitizen is stored as a number, but it only has two values (0 or 1), so we treat it as a category.

<aside>
📊

On a high level, mutual information is a value that measures how much knowing one variable tells you about another. If two variables have nothing to do with each other, their mutual information is 0.

For example, if `X` is the result of spinning one roulette wheel and `Y` is the result of spinning a different wheel, the mutual information between `X` and `Y` is 0. But if `X` is the color on a roulette wheel and `Y` is the number, they share some information, because only certain numbers have certain colors.

</aside>

In our case: if I know a customer's value in this column, how much do I learn about whether they will churn? 0 means we learn nothing. The higher the number, the more we learn.

In [30]:
from sklearn.metrics import mutual_info_score

mutual_info_score(df_train.churn, df_train.contract)
# Output: 0.0966515250997

mutual_info_score(df_train.churn, df_train.gender)
# Output: 6.877785208564413e-05

6.877785208564413e-05

`e-05` means "move the decimal point 5 places to the left", so the score for gender is 0.00007, almost zero. Knowing a customer's gender tells us almost nothing about churn, while knowing the contract tells us a lot.

Now let's calculate the score for all categorical columns at once. The `apply` function runs a function on every column, but it gives the function only **one** column at a time. `mutual_info_score` needs two inputs (the column and churn), so we wrap it in a small function that takes one column.

In [31]:
def mutual_info_churn_score(series):
    return mutual_info_score(series, df_train.churn)

mi = df_train[categorical].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)
# Output:

contract            0.096652
onlinesecurity      0.063393
techsupport         0.060935
internetservice     0.053313
onlinebackup        0.045424
paymentmethod       0.042861
deviceprotection    0.042007
streamingtv         0.030844
streamingmovies     0.030705
paperlessbilling    0.019077
dependents          0.014129
partner             0.010227
seniorcitizen       0.010059
multiplelines       0.000654
gender              0.000069
phoneservice        0.000040
dtype: float64

`sort_values(ascending=False)` puts the biggest scores first. The contract is the most informative column, followed by the internet-related services. Gender, phone service and multiple lines are close to zero, so they tell us very little about churn.

### Correlation

For columns that hold numbers we use the **correlation coefficient** (Pearson's correlation). It answers one question: when this number goes up, does churn go up or down? The answer, usually written as `r`, is between -1 and 1:

- `r` close to **1**: when the number goes up, churn goes up.
- `r` close to **-1**: when the number goes up, churn goes down.
- `r` close to **0**: there is no clear relationship.

The further `r` is from 0 (in either direction), the more important the column.

In [32]:
df_train[numerical].corrwith(df_train.churn)

tenure           -0.344925
monthlycharges    0.188574
totalcharges     -0.193370
dtype: float64

corrwith calculates the correlation of each numerical column with churn.

- `tenure` is negative and the strongest: the longer someone has been a customer, the less likely they are to leave.
- `monthlycharges` is positive: customers who pay more each month leave a bit more often.
- `totalcharges` is negative: customers who have paid more in total have usually stayed longer, so they leave less.

Let's check the tenure result with real groups of customers.

In [33]:
df_train[df_train.tenure <= 2].churn.mean()
# Output: np.float64(0.5863636363636363)

df_train[df_train.tenure > 12].churn.mean()
# Output: np.float64(0.17205957883923986)

np.float64(0.17205957883923986)

Almost 59% of customers in their first two months leave, compared with about 17% of customers who have stayed for more than a year. That is exactly what the negative correlation was telling us.

In [35]:
df.to_csv('Section 1 autumn 2027telco_churn_clean.csv', index=False)